In [ ]:
#Analiza SEA pormieniowanie kosmiczne -> trzęsienia ziemi
#1. Filtrowanie trzęsień ziemi wg magnitudy
#2. Deklasteryzacja metodą Gardnera-Knopoffa
#3. Impulsy definiowane na |ΔCR|
#4. Deklasteryzacja IMPULSÓW
#5. Analiza SEA (Superposed Epoch Analysis)

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


HOME_PATH = Path.home()
PROJECT_DIR = HOME_PATH / "Desktop" / "kosmosejsmiczne"
DATA_DIR = PROJECT_DIR / "dane"
SEA_PLOTS_DIR = PROJECT_DIR / "SEAplots"

MAG_THRESHOLD = 4.5
FS_TIME_PROP = 1.0           # parametr okna foreshocków (Gardner-Knopoff, wg OpenQuake)

# Parametry SEA
WINDOW_BEFORE = 30
WINDOW_AFTER = 30
N_BOOTSTRAP = 1000
THRESHOLD_PERCENTILE = 0.98

MIN_GAP_DAYS = 5

RAW_QUAKES_FILE = DATA_DIR / "eq_data.csv"
FILTERED_FILE = DATA_DIR / f"eq_data_m{int(MAG_THRESHOLD * 10)}_plus.csv"
DECLUSTERED_FILE = DATA_DIR / f"eq_data_declustered_m{int(MAG_THRESHOLD * 10)}.csv"
COSMIC_FILE = DATA_DIR / "moskwadane.csv"

# Nazwa wyjsciowy
_THR_PCT = int(round(THRESHOLD_PERCENTILE * 100))
SEA_PLOT = SEA_PLOTS_DIR / f"SEAthr{_THR_PCT}mag{MAG_THRESHOLD}_deltaCR.png"


# filtrowanie magntud
def filter_magnitude(input_file: Path, output_file: Path, min_mag: float = MAG_THRESHOLD) -> pd.DataFrame:
    print("--- ETAP 1: Filtrowanie magnitud ---")

    column_names = ["time", "lat", "long", "mag"]
    df = pd.read_csv(input_file, names=column_names, header=None)

    # Konwersja 'mag'
    df["mag"] = pd.to_numeric(df["mag"], errors="coerce")

    df["time"] = pd.to_datetime(df["time"].astype(str).str.replace("Z", ""), errors="coerce")
    df = df.dropna(subset=["time", "mag"])

    df_filtered = df[df["mag"] >= min_mag].copy()
    df_filtered = df_filtered.sort_values(by="time").reset_index(drop=True)

    df_filtered.to_csv(output_file, index=False)

    print(f"liczba trzęsień: (M >= {min_mag}): {len(df_filtered)}")
    print(f"nazwa pliku: {output_file}\n")

    return df_filtered


#  deklasteryzacja Gardnera-Knopoffa
def haversine_distance(lat1, lon1, lat2, lon2):
    #Odległość na sferze
    R_earth = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R_earth * c


def get_gk_window(magnitude):
    #Okno czasowo-przestrzenne Gardnera-Knopoffa dla danej magnitudy.
    magnitude = np.asarray(magnitude, dtype=float)
    r_km = 10 ** (0.1238 * magnitude + 0.983)

    t_days = np.where(
        magnitude >= 6.5,
        10 ** (0.032 * magnitude + 2.7389),
        10 ** (0.5409 * magnitude - 0.547),
    )
    return r_km, t_days


def decluster_gk(input_file: Path, output_file: Path) -> pd.DataFrame:

    df = pd.read_csv(input_file)
    df["time"] = pd.to_datetime(df["time"])
    total_records = len(df)

    # konwersja kolumn na tablice numpy dla szybszego przetwarzania
    times_sec = df["time"].values.astype("datetime64[s]").astype(np.int64)
    lats = df["lat"].to_numpy(dtype=float)
    lons = df["long"].to_numpy(dtype=float)
    mags = df["mag"].to_numpy(dtype=float)

    #  od największej magnitudy do najmniejszej
    process_order = np.argsort(-mags)

    # Osobny indeks posortowany po czasie — ogranicza tylko do zdarzen lezacych
    # w oknie czasowym, zamiast liczyc w calym katalogu
    time_order = np.argsort(times_sec)
    sorted_times = times_sec[time_order]

    r_max_arr, t_max_arr = get_gk_window(mags)

    is_aftershock = np.zeros(total_records, dtype=bool)
    DAY_SEC = 24 * 3600
    report_every = max(total_records // 10, 1)

    for count, i in enumerate(process_order):
        if is_aftershock[i]:
            continue

        if count % report_every == 0:
            print(f"  ...przetworzono {count}/{total_records} zdarzeń")

        r_max, t_max = r_max_arr[i], t_max_arr[i]

        time_low = times_sec[i] - t_max * FS_TIME_PROP * DAY_SEC   # wykluczone (>)
        time_high = times_sec[i] + t_max * DAY_SEC                 # włączone (<=)

        left = np.searchsorted(sorted_times, time_low, side="right")
        right = np.searchsorted(sorted_times, time_high, side="right")

        if left >= right:
            continue

        candidate_idx = time_order[left:right]
        candidate_idx = candidate_idx[candidate_idx != i]  # bez samego siebie

        if candidate_idx.size == 0:
            continue

        distances = haversine_distance(
            lats[i], lons[i],
            lats[candidate_idx], lons[candidate_idx],
        )

        linked_idx = candidate_idx[distances <= r_max]
        is_aftershock[linked_idx] = True

    df_clean = df[~is_aftershock].copy()
    df_clean = df_clean.sort_values(by="time").reset_index(drop=True)

    df_clean.to_csv(output_file, index=False)

    print(f"Liczba wejściowa trzęsień: {len(df)}")
    print(f"Liczba niezależnych wstrząsów głównych: {len(df_clean)}")

    return df_clean


# deklasteryzacja impulsow |ΔCR|
def decluster_impulses(candidate_days, strength: pd.Series, min_gap_days: int,
                        verbose: bool = True, csv_output: Path = None) -> list:

    sorted_days = sorted(candidate_days)
    if len(sorted_days) == 0:
        return []

    clusters = [[sorted_days[0]]]
    for day in sorted_days[1:]:
        gap = (day - clusters[-1][-1]).days
        if gap < min_gap_days:
            clusters[-1].append(day)
        else:
            clusters.append([day])

    representatives = [
        max(cluster, key=lambda d: strength.loc[d])
        for cluster in clusters
    ]

    n_single = sum(1 for c in clusters if len(c) == 1)
    n_multi = len(clusters) - n_single
    print(f"Deklasteryzacja impulsów: {len(sorted_days)} kandydatów -> "
          f"{len(clusters)} klastrów -> {len(representatives)} reprezentantów "
          f"(min_gap={min_gap_days} dni)")
    print(f"  Klastry jednodniowe: {n_single}  |  Klastry wielodniowe: {n_multi} "
          f"({100 * n_multi / len(clusters):.1f}% klastrów)")
    if n_multi == 0:
        print("  -> WSZYSTKIE klastry jednodniowe: deklasteryzacja nic tu nie zmienia,")
        print("     poza kosmetyką (można by ją pominąć bez wpływu na wynik).")
    else:
        print(f"  -> {n_multi} klaster(ów) miało więcej niż 1 dzień -- deklasteryzacja")
        print("     zapobiegła policzeniu tego samego epizodu wielokrotnie.")

    if verbose:
        for cluster_id, (cluster, rep) in enumerate(zip(clusters, representatives), start=1):
            print(f"  Klaster {cluster_id}/{len(clusters)} ({len(cluster)} dni) "
                  f"-> reprezentant: {rep}  wartość={strength.loc[rep]:.3f}")
        print()

    if csv_output is not None:
        detail_rows = []
        for cluster_id, (cluster, rep) in enumerate(zip(clusters, representatives), start=1):
            for day in cluster:
                detail_rows.append({
                    "klaster": cluster_id,
                    "dzien": day,
                    "wartosc": strength.loc[day],
                    "czy_reprezentant": day == rep,
                })
        detail_df = pd.DataFrame(detail_rows)
        csv_output.parent.mkdir(parents=True, exist_ok=True)
        detail_df.to_csv(csv_output, index=False)
        print(f"Pełny rozkład klastrów zapisany w: {csv_output}\n")

    return representatives


# Analiza SEA (Superposed Epoch Analysis)
def run_sea_analysis(quakes_file: Path, cosmic_file: Path, output_plot: Path):

    # Trzęsienia — dzienna liczba zdarzeń
    quakes_df = pd.read_csv(quakes_file)
    quakes_df["time"] = pd.to_datetime(quakes_df["time"])
    daily_quakes = quakes_df.groupby(quakes_df["time"].dt.date).size()

    # Promieniowanie kosmiczne — dzienna średnia
    cosmic_raw = pd.read_csv(cosmic_file, sep=r"\s+|,", engine="python", header=None)
    date_series = cosmic_raw.iloc[:, 0].astype(str) + " " + cosmic_raw.iloc[:, 1].astype(str)
    cosmic_dates = pd.to_datetime(date_series, errors="coerce")
    cosmic_values = pd.to_numeric(cosmic_raw.iloc[:, 2], errors="coerce")
    cosmic_df = pd.DataFrame({"datetime": cosmic_dates, "value": cosmic_values}).dropna()
    daily_cosmic = cosmic_df.groupby(cosmic_df["datetime"].dt.date)["value"].mean()

    # Wspólny zakres dat
    start_date = max(daily_quakes.index.min(), daily_cosmic.index.min())
    end_date = min(daily_quakes.index.max(), daily_cosmic.index.max())
    full_range = pd.date_range(start=start_date, end=end_date, freq="D").date

    daily_counts = daily_quakes.reindex(full_range, fill_value=0)
    daily_cosmic = daily_cosmic.reindex(full_range).dropna()


    delta_cr = daily_cosmic.diff().abs().dropna()

    threshold = delta_cr.quantile(THRESHOLD_PERCENTILE)
    candidate_days = delta_cr[delta_cr >= threshold].index
    print(f"Znaleziono {len(candidate_days)} kandydatów na impulsy (|ΔCR| >= próg, przed deklasteryzacją)")

    # Deklasteryzacja — na wypadek, gdyby jeden epizod Forbusha rozciągał
    # się na kilka sąsiednich dni z rzędu (dwuetapowy profil spadku)
    key_days = decluster_impulses(candidate_days, delta_cr, MIN_GAP_DAYS)

    lags = np.arange(-WINDOW_BEFORE, WINDOW_AFTER + 1)
    counts_array = daily_counts.values

    key_indices = []
    for day in key_days:
        if day in daily_counts.index:
            idx = daily_counts.index.get_loc(day)
            if WINDOW_BEFORE <= idx < len(counts_array) - WINDOW_AFTER:
                key_indices.append(idx)

    key_indices = np.array(key_indices)
    n_events = len(key_indices)
    print(f"Impulsów w analizie (po deklasteryzacji): {n_events}")

    if n_events == 0:
        raise ValueError("Brak impulsów do analizy!")

    matrix_raw = np.array([
        counts_array[idx - WINDOW_BEFORE: idx + WINDOW_AFTER + 1]
        for idx in key_indices
    ])

    # Standaryzacja: odjęcie tła - sredniej z dni przed impulsem
    baseline_means = np.mean(matrix_raw[:, :WINDOW_BEFORE], axis=1, keepdims=True)
    matrix_events = matrix_raw - baseline_means
    sea_mean = np.mean(matrix_events, axis=0)

    # Bootstrap — pula losowań nie jest deklastrowana celowo
    .
    print(f"Monte Carlo: {N_BOOTSTRAP} symulacji...")
    min_idx = WINDOW_BEFORE
    max_idx = len(counts_array) - WINDOW_AFTER - 1
    possible_indices = np.arange(min_idx, max_idx + 1)

    np.random.seed(42)
    bootstrap_means = np.zeros((N_BOOTSTRAP, len(lags)))

    for i in range(N_BOOTSTRAP):
        rand_idx = np.random.choice(possible_indices, size=n_events, replace=False)

        rand_matrix_raw = np.array([
            counts_array[idx - WINDOW_BEFORE: idx + WINDOW_AFTER + 1]
            for idx in rand_idx
        ])

        rand_baseline = np.mean(rand_matrix_raw[:, :WINDOW_BEFORE], axis=1, keepdims=True)
        rand_matrix_norm = rand_matrix_raw - rand_baseline
        bootstrap_means[i, :] = np.mean(rand_matrix_norm, axis=0)

    ci_lower = np.percentile(bootstrap_means, 2.5, axis=0)
    ci_upper = np.percentile(bootstrap_means, 97.5, axis=0)

    # Wykres

    fig, ax = plt.subplots(figsize=(14, 7))

    ax.fill_between(lags, ci_lower, ci_upper, color="gray", alpha=0.3,
                     label="95% przedział ufności (szum)")
    ax.axhline(y=0, color="black", linestyle="--", linewidth=1)
    ax.axvline(x=0, color="blue", linestyle=":", label="Dzień impulsu")
    ax.plot(lags, sea_mean, color="red", linewidth=2.5, marker="o",
            markersize=4, label=f"Odchylenie liczby trzęsień (M>={MAG_THRESHOLD})")

    anomalies = (sea_mean < ci_lower) | (sea_mean > ci_upper)
    if np.any(anomalies):
        ax.scatter(lags[anomalies], sea_mean[anomalies],
                   color="green", s=80, zorder=5, label="Anomalia (p < 0.05)")

    top_pct = (1 - THRESHOLD_PERCENTILE) * 100  # % dni o najwyższych wartościach |ΔCR| uznanych za impulsy

    ax.set_title(
        f"SEA analysis (impulsy = |ΔCR|), M>{MAG_THRESHOLD}, impulsy = {n_events} "
        f"(zdeklastrowane, min. odstęp {MIN_GAP_DAYS}d)",
        fontsize=13,
    )
    ax.set_xlabel("Lag [dni od impulsu]", fontsize=12)
    ax.set_ylabel("Zmiana średniej dziennej liczby trzęsień", fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(loc="lower left", fontsize=10)
    ax.text(
        0.02, 0.97,
        f"Górne {top_pct:.0f}% dni wg |ΔCR| ({len(candidate_days)} kandydatów -> {len(key_days)} po "
        f"deklasteryzacji, z {len(delta_cr)} dni w zakresie danych)",
        transform=ax.transAxes,
        fontsize=10,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="gray", alpha=0.85),
    )

    plt.tight_layout()
    output_plot.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_plot, dpi=300, bbox_inches="tight")
    plt.close()

    # Raport

    print("RAPORT")
    print(f"Impulsy = górne {top_pct:.0f}% dni wg |ΔCR| (zmiana dzień-do-dnia), "
          f"po deklasteryzacji: {len(candidate_days)} -> {len(key_days)} "
          f"(próg = {threshold:.3f}, min. odstęp = {MIN_GAP_DAYS} dni)")
    print(f"Impulsów w analizie: {n_events}")
    print(f"Średnia liczba trzęsień w tle: {np.mean(matrix_raw[:, :WINDOW_BEFORE]):.3f}")
    print(f"Wykres w: {output_plot}\n")


# uruchomienie
def main():
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(SEA_PLOTS_DIR, exist_ok=True)

    filter_magnitude(RAW_QUAKES_FILE, FILTERED_FILE, MAG_THRESHOLD)
    decluster_gk(FILTERED_FILE, DECLUSTERED_FILE)
    run_sea_analysis(DECLUSTERED_FILE, COSMIC_FILE, SEA_PLOT)


if __name__ == "__main__":
    main()